# 16 — Génération routée et citations Facebook

Ce dernier test avant le notebook final charge le pipeline complet avec deux index séparés. Six questions vérifient les routes Facebook, mixte et institutionnelle, la qualité des réponses, les citations, le cache et l’observabilité.

Aucun appel Apify et aucune réindexation ne sont effectués. Ollama doit être démarré.

In [1]:
from pathlib import Path
import json
import sys
import time
import unicodedata
import re

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('Projet :', PROJECT_ROOT)

Projet : c:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs\RAG ENSEA


In [2]:
import importlib
import rag_pipeline as rag_pipeline_module
from config import load_config

rag_pipeline_module = importlib.reload(rag_pipeline_module)
RAGPipeline = rag_pipeline_module.RAGPipeline
config = load_config(PROJECT_ROOT / 'config.yaml')
pipeline = RAGPipeline(config)

print('Chunks institutionnels :', pipeline.store.dense.count())
print('Chunks Facebook :', pipeline.facebook_store.dense.count() if pipeline.facebook_store else 0)
print('Routage Facebook activé :', config.facebook_scraping.retrieval_enabled)
assert pipeline.store.dense.count() == 4459
assert pipeline.facebook_store is not None and pipeline.facebook_store.dense.count() == 250

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Chunks institutionnels : 4459
Chunks Facebook : 250
Routage Facebook activé : True


In [3]:
test_cases = [
    {
        'id': 'fb-cdr',
        'question': "Qui est la sixième et dernière finaliste du concours ENSEA d'éloquence ?",
        'expected_route': 'facebook',
        'keywords': ['zamblé', 'chimène'],
        'expected_domain': 'facebook.com',
    },
    {
        'id': 'fb-edsc',
        'question': 'Quel webinaire du Data Science Club porte sur Kaggle de A à Z ?',
        'expected_route': 'facebook',
        'keywords': ['kaggle'],
        'expected_domain': 'facebook.com',
    },
    {
        'id': 'fb-ecostat',
        'question': 'Où lire la Dépêche EcoStat numéro 20 ?',
        'expected_route': 'facebook',
        'keywords': ['20'],
        'expected_domain': 'facebook.com',
    },
    {
        'id': 'fb-aes',
        'question': "Selon l'AES, quand ont commencé les campagnes de renouvellement des clubs et associations ?",
        'expected_route': 'facebook',
        'keywords': ['9 juin 2025'],
        'expected_domain': 'facebook.com',
    },
    {
        'id': 'both-cisea',
        'question': 'Quel appel à communications a été publié pour CISEA 2026 ?',
        'expected_route': 'both',
        'keywords': ['cisea', '2026'],
        'expected_domain': None,
    },
    {
        'id': 'inst-master-ds',
        'question': "Qu'est-ce que le Master en Data Science proposé par l'ENSEA ?",
        'expected_route': 'institutional',
        'keywords': ['inp-hb', 'polytechnique'],
        'expected_domain': 'ensea',
    },
]
print('Questions de génération :', len(test_cases))

Questions de génération : 6


In [4]:
def normalize_text(text):
    text = unicodedata.normalize('NFKD', str(text).casefold())
    return ''.join(char for char in text if not unicodedata.combining(char))

generation_rows = []
for index, case in enumerate(test_cases, 1):
    started = time.perf_counter()
    response = pipeline.ask(case['question'])
    elapsed = time.perf_counter() - started
    decision = pipeline.retriever.last_decision
    citation_urls = [citation.url for citation in response.citations]
    normalized_answer = normalize_text(response.answer)
    keyword_hits = [
        keyword for keyword in case['keywords']
        if normalize_text(keyword) in normalized_answer
    ]
    if case['expected_domain'] == 'facebook.com':
        citation_source_ok = any('facebook.com' in url for url in citation_urls)
    elif case['expected_domain'] == 'ensea':
        citation_source_ok = bool(citation_urls) and all('facebook.com' not in url for url in citation_urls)
    else:
        citation_source_ok = bool(citation_urls)
    row = {
        **case,
        'actual_route': decision.route,
        'route_reason': decision.reason,
        'answer': response.answer,
        'citation_count': len(response.citations),
        'citation_urls': citation_urls,
        'citation_source_ok': citation_source_ok,
        'keyword_coverage': len(keyword_hits) / len(case['keywords']),
        'keyword_hits': keyword_hits,
        'retrieval_seconds': pipeline.retriever.last_timings.get('total_seconds', 0.0),
        'generation_seconds': pipeline.generator.last_generation_stats.get('total_seconds', 0.0),
        'total_seconds': elapsed,
        'cache_hit': pipeline.generator.last_generation_stats.get('cache_hit', False),
    }
    generation_rows.append(row)
    print(f"[{index}/{len(test_cases)}] {case['id']} | route={decision.route} | citations={len(response.citations)} | total={elapsed:.2f}s")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[1/6] fb-cdr | route=facebook | citations=1 | total=36.10s
[2/6] fb-edsc | route=facebook | citations=1 | total=6.84s
[3/6] fb-ecostat | route=facebook | citations=1 | total=9.10s
[4/6] fb-aes | route=facebook | citations=1 | total=7.05s
[5/6] both-cisea | route=both | citations=1 | total=13.53s
[6/6] inst-master-ds | route=institutional | citations=3 | total=130.04s


In [5]:
import pandas as pd

generation_df = pd.DataFrame(generation_rows)
generation_metrics = pd.Series({
    'questions': len(generation_df),
    'route_accuracy': (generation_df['actual_route'] == generation_df['expected_route']).mean(),
    'citation_rate': generation_df['citation_count'].gt(0).mean(),
    'citation_source_accuracy': generation_df['citation_source_ok'].mean(),
    'mean_keyword_coverage': generation_df['keyword_coverage'].mean(),
    'mean_retrieval_seconds': generation_df['retrieval_seconds'].mean(),
    'mean_generation_seconds': generation_df['generation_seconds'].mean(),
    'mean_total_seconds': generation_df['total_seconds'].mean(),
})
display(generation_metrics.to_frame('valeur'))
display(generation_df[[
    'id', 'expected_route', 'actual_route', 'citation_count',
    'citation_source_ok', 'keyword_coverage', 'retrieval_seconds',
    'generation_seconds', 'answer', 'citation_urls',
]])

,valeur
questions,6.000000
route_accuracy,1.000000
citation_rate,1.000000
citation_source_accuracy,1.000000
mean_keyword_coverage,1.000000
mean_retrieval_seconds,1.120750
mean_generation_seconds,32.640326
mean_total_seconds,33.776374


,id,expected_route,actual_route,citation_count,citation_source_ok,keyword_coverage,retrieval_seconds,generation_seconds,answer,citation_urls
0,fb-cdr,facebook,facebook,1,True,1.0,3.175677,32.899621,La sixième et dernière finaliste du concours E...,[https://www.facebook.com/cdr.ensea.1/posts/pf...
1,fb-edsc,facebook,facebook,1,True,1.0,0.466582,6.366672,Le webinaire du Data Science Club qui porte su...,[https://www.facebook.com/edsc.ensea/posts/pfb...
2,fb-ecostat,facebook,facebook,1,True,1.0,0.272247,8.822665,La Dépêche EcoStat numéro 20 peut être lue et ...,[https://www.facebook.com/DepecheEcostat/posts...
3,fb-aes,facebook,facebook,1,True,1.0,0.402150,6.637117,Les campagnes de renouvellement des clubs et a...,[https://www.facebook.com/permalink.php?story_...
4,both-cisea,both,both,1,True,1.0,2.018373,11.501876,L'appel à communications pour CISEA 2026 a été...,[https://www.facebook.com/ensea.ci/posts/pfbid...
5,inst-master-ds,institutional,institutional,3,True,1.0,0.389468,129.614007,Le Master en Data Science proposé par l'ENSEA ...,[https://ensea.ed.ci/formation-initiale/master...


## Cache exact et observabilité

La première question est répétée : la génération doit être instantanément servie depuis le cache, tout en conservant le même routage.

In [6]:
cache_started = time.perf_counter()
cache_response = pipeline.ask(test_cases[0]['question'])
cache_total_seconds = time.perf_counter() - cache_started
cache_stats = {
    'cache_hit': pipeline.generator.last_generation_stats.get('cache_hit'),
    'generation_seconds': pipeline.generator.last_generation_stats.get('total_seconds'),
    'total_seconds': cache_total_seconds,
    'route': pipeline.retriever.last_decision.route,
    'citation_count': len(cache_response.citations),
}
cache_stats

{'cache_hit': True,
 'generation_seconds': 0.0,
 'total_seconds': 0.359918000000107,
 'route': 'facebook',
 'citation_count': 1}

In [7]:
from observability import load_traces

all_traces = load_traces(config.observability.traces_full_path)
tested_questions = {case['question']: case['expected_route'] for case in test_cases}
recent_traces = [
    event for event in all_traces
    if event.get('question') in tested_questions and event.get('event_type') == 'rag'
]
latest_by_question = {}
for event in recent_traces:
    latest_by_question[event['question']] = event
trace_rows = [
    {
        'question': question,
        'expected_route': expected_route,
        'observed_route': latest_by_question.get(question, {}).get('metadata', {}).get('route'),
        'status': latest_by_question.get(question, {}).get('status'),
        'citation_count': latest_by_question.get(question, {}).get('citation_count'),
    }
    for question, expected_route in tested_questions.items()
]
trace_df = pd.DataFrame(trace_rows)
display(trace_df)

,question,expected_route,observed_route,status,citation_count
0,Qui est la sixième et dernière finaliste du co...,facebook,facebook,ok,1
1,Quel webinaire du Data Science Club porte sur ...,facebook,facebook,ok,1
2,Où lire la Dépêche EcoStat numéro 20 ?,facebook,facebook,ok,1
3,"Selon l'AES, quand ont commencé les campagnes ...",facebook,facebook,ok,1
4,Quel appel à communications a été publié pour ...,both,both,ok,1
5,Qu'est-ce que le Master en Data Science propos...,institutional,institutional,ok,3


In [8]:
assert (generation_df['actual_route'] == generation_df['expected_route']).all()
assert generation_df['citation_count'].gt(0).all(), 'Une réponse ne contient aucune citation.'
assert generation_df['citation_source_ok'].all(), 'Une citation vient du mauvais index.'
assert cache_stats['cache_hit'] is True and cache_stats['generation_seconds'] == 0.0
assert cache_stats['route'] == 'facebook'
assert trace_df['status'].eq('ok').all()
assert (trace_df['observed_route'] == trace_df['expected_route']).all()
print('Pipeline routé, citations, cache et observabilité validés.')

result_path = PROJECT_ROOT / 'data' / 'experiments' / 'routed_generation_benchmark.json'
result_path.write_text(json.dumps({
    'metrics': generation_metrics.to_dict(),
    'rows': generation_rows,
    'cache': cache_stats,
    'traces': trace_rows,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Résultats :', result_path)

Pipeline routé, citations, cache et observabilité validés.
Résultats : c:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs\RAG ENSEA\data\experiments\routed_generation_benchmark.json


## Arrêt de validation

Sauvegardez le notebook et transmettez les métriques, les réponses et toute assertion éventuelle. Si ce test passe, la construction du notebook final `rag_ensea.ipynb` pourra commencer.